<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Stream Performance Metrics

This tutorial measures packet latency, stalls, bubbles, gaps, and sustained throughput from timestamped monitor frames. Metric calculations are protocol-independent, but clock calibration and real timestamps normally come from a cocotb simulation.

In [1]:
from fpga_verification.sim import (
    PacketMetrics,
    PacketObservation,
    PacketSequenceMetrics,
    StreamPerformanceAnalyzer,
)
from cocotbext.avalon import AvalonSTFrame

## 1. Observation model

`PacketObservation` pairs the same logical packet at the input and output of a DUT. `beats` is the number of useful transfers in the packet. Both frames must expose `sim_time_start` and `sim_time_end`; `AvalonSTFrame` receives these timestamps from a source, sink, or monitor.

```text
input SoP ───── input EoP
       └── DUT/pipeline ──┐
                    output SoP ───── output EoP
```

In [2]:
input_frame = AvalonSTFrame([0x10, 0x20, 0x30, 0x40])
output_frame = AvalonSTFrame([0x10, 0x20, 0x30, 0x40])

# These fields are normally filled by simulation monitors.
input_frame.sim_time_start = 100
input_frame.sim_time_end = 130
output_frame.sim_time_start = 120
output_frame.sim_time_end = 160

observation = PacketObservation(
    name="frame_0",
    beats=4,
    input_frame=input_frame,
    output_frame=output_frame,
)
observation

PacketObservation(name='frame_0', beats=4, input_frame=AvalonSTFrame(data=[0x10, 0x20, 0x30, 0x40], channel=None, error=None, empty=None, size=4, sim_time_start=100, sim_time_end=130), output_frame=AvalonSTFrame(data=[0x10, 0x20, 0x30, 0x40], channel=None, error=None, empty=None, size=4, sim_time_start=120, sim_time_end=160))

## 2. Clock calibration

Metrics are expressed in clock cycles rather than simulator time steps. Create the analyzer with `await StreamPerformanceAnalyzer.from_clock(dut.clk)` after the clock has started. It observes two rising edges and stores the measured period.

```python
analyzer = await StreamPerformanceAnalyzer.from_clock(dut.clk)
```

Timestamp differences must be exact multiples of this period. This catches measurements taken from unrelated clocks or unaligned events.

## 3. PacketMetrics

`analyzer.packet(observation)` returns `PacketMetrics`:

| Field | Meaning |
|---|---|
| `input_cycles`, `output_cycles` | Inclusive packet duration |
| `input_stalls` | Input duration minus useful beats |
| `output_bubbles` | Output duration minus useful beats |
| `input_efficiency`, `output_efficiency` | Useful beats divided by duration |
| `sop_latency` | Input SoP to output SoP |
| `eop_latency` | Input EoP to output EoP |
| `end_to_end_cycles` | Input SoP through output EoP, inclusive |

Use `metrics.log(logger)` to write a formatted summary to a Python or cocotb logger.

In [3]:
async def measure_one_packet(clock, input_monitor, output_monitor, beats):
    analyzer = await StreamPerformanceAnalyzer.from_clock(clock)
    input_seen = await input_monitor.recv()
    output_seen = await output_monitor.recv()

    observation = PacketObservation(
        name="video",
        beats=beats,
        input_frame=input_seen,
        output_frame=output_seen,
    )
    return analyzer.packet(observation)

## 4. PacketSequenceMetrics

`analyzer.sequence(name, observations)` analyzes an ordered sequence and returns `PacketSequenceMetrics`. In addition to the individual `PacketMetrics`, it reports packet gaps, SoP intervals, boundary overlap, the maximum number of packets in flight, and aggregate efficiency.

`sustainable_efficiency` is the smaller of input and output efficiency. `required_clock_multiplier` is the clock-rate multiplier required to match an ideal one-beat-per-cycle stream.

The assertion helpers turn performance requirements into regression checks:

```python
sequence.assert_input_packet_gap_at_most(2)
sequence.assert_all_boundaries_overlap()
```

In [4]:
async def measure_sequence(clock, observations):
    analyzer = await StreamPerformanceAnalyzer.from_clock(clock)
    metrics = analyzer.sequence("video_pipeline", observations)

    metrics.assert_input_packet_gap_at_most(2)
    print(f"Input efficiency: {metrics.input_efficiency:.2%}")
    print(f"Output efficiency: {metrics.output_efficiency:.2%}")
    print(f"Maximum packets in flight: {metrics.max_packets_in_flight}")
    return metrics

## 5. Typical workflow

1. Start the DUT clock and calibrate one analyzer.
2. Capture matching input and output `AvalonSTFrame` objects.
3. Determine the useful beat count from the packet format.
4. Build `PacketObservation` objects in input order.
5. Calculate per-packet or sequence metrics.
6. Log the result and assert the performance contract.

Do not use list length blindly when the final beat contains empty symbols: `beats` counts accepted transfers, not payload symbols or bytes.